# Factor Analysis for Experiments 1, 2, and 3

This notebook reproduces the preprocessing used in:
- `Experiment1/analysis/analysis-0306.ipynb`
- `Experiment2/analysis/0325.ipynb`
- `Experiment3/analysis/analysis-0318.ipynb`

Then it runs fixed-size factor analysis with **1**, **2**, and **4** factors for both:
- trust/usability questions
- interaction-experience questions

Final outputs:
- one combined loading table containing both trust and interaction questions
- hierarchical columns: Exp1/Exp2/Exp3 -> 1-factor/2-factor/4-factor -> factor columns
- values with `abs(loading) < 0.3` are excluded
- a LaTeX export of the combined table in `factor.tex`


In [21]:
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import FactorAnalysis

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", None)


In [22]:
TRUST_QUESTION_KEYS = [
    "skeptical_rating",
    "data_trust",
    "usability_difficulty",
    "comprehension_ease",
]

INTERACTION_QUESTION_KEYS = [
    "navigation_control",
    "content_control",
    "pace_control",
    "interface_exploration",
    "interface_responsiveness",
    "user_communication",
    "personal_conversation",
    "interface_interaction",
    "interface_sensitivity",
]

TRUST_PROMPT_MAP = {
    "skeptical_rating": "I was skeptical of the visualization output.",
    "data_trust": "I trusted the data shown by the visualization.",
    "usability_difficulty": "The visualization was difficult to use.",
    "comprehension_ease": "The visualization made the task easier to understand.",
}

INTERACTION_PROMPT_MAP = {
    "navigation_control": "I was in control of my navigation through this interface.",
    "content_control": "I had control over the content I wanted to inspect.",
    "pace_control": "I was in control of the pace of my visit to this interface.",
    "interface_exploration": "The interface supported the way I wanted to explore the visualization.",
    "interface_responsiveness": "The interface responded quickly and efficiently to what I wanted to inspect.",
    "user_communication": "The interface supported communication with other users who shared my interests.",
    "personal_conversation": "Interacting with the interface felt like a personal conversation.",
    "interface_interaction": "I felt that I could interact meaningfully with the interface.",
    "interface_sensitivity": "The interface felt sensitive to my actions.",
}

FACTOR_COUNTS = [1, 2, 4]
LOADING_THRESHOLD = 0.3


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Experiment1").exists() and (candidate / "Experiment2").exists() and (candidate / "Experiment3").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing Experiment1/Experiment2/Experiment3")


PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")


Project root: /Users/songwen/Documents/Code Lab/Interact4Trust


In [23]:
def parse_response_json(value):
    if pd.isna(value):
        return {}
    if isinstance(value, dict):
        return value
    try:
        parsed = json.loads(value)
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        return {}


def expand_response_columns(data, response_col="response"):
    if response_col not in data.columns:
        return data.copy()

    parsed = data[response_col].apply(parse_response_json)
    expanded = pd.json_normalize(parsed)

    merged = data.reset_index(drop=True).copy()
    for col in expanded.columns:
        parsed_values = expanded[col]
        if col in merged.columns:
            merged[col] = merged[col].where(merged[col].notna(), parsed_values)
        else:
            merged[col] = parsed_values

    return merged


def ensure_likert_columns(data, keys):
    if data.empty:
        return data.copy()

    working = data.copy()
    parsed = (
        working["response"].apply(parse_response_json)
        if "response" in working.columns
        else pd.Series([{}] * len(working), index=working.index)
    )

    for key in keys:
        existing = (
            pd.to_numeric(working[key], errors="coerce")
            if key in working.columns
            else pd.Series(np.nan, index=working.index)
        )

        if key not in working.columns or existing.notna().sum() == 0:
            extracted = pd.to_numeric(parsed.apply(lambda d: d.get(key, np.nan)), errors="coerce")
            valid = extracted.dropna()
            if not valid.empty and (valid >= 0).all() and (valid <= 6).all():
                extracted = extracted + 1
            working[key] = extracted
        else:
            working[key] = existing

    return working


In [24]:
NORMALIZED_FLAG_COLUMN = "_cousineau_normalized"
MOREY_FACTOR_COLUMN = "_morey_factor"
K_CONDITIONS_COLUMN = "_k_conditions"


def identify_attention_check_failures(
    data,
    trust_keys=TRUST_QUESTION_KEYS,
    interaction_keys=INTERACTION_QUESTION_KEYS,
    survey_trial_type="trust-survey",
):
    if data is None or data.empty:
        return set(), pd.DataFrame()

    if "trial_type" not in data.columns or "participant_id" not in data.columns:
        return set(), pd.DataFrame()

    survey_rows = data[data["trial_type"] == survey_trial_type].copy()
    if survey_rows.empty:
        return set(), pd.DataFrame()

    survey_rows = expand_response_columns(survey_rows)
    parsed_response = (
        survey_rows["response"].apply(parse_response_json)
        if "response" in survey_rows.columns
        else pd.Series([{}] * len(survey_rows), index=survey_rows.index)
    )

    for key in trust_keys + interaction_keys:
        existing = (
            pd.to_numeric(survey_rows[key], errors="coerce")
            if key in survey_rows.columns
            else pd.Series(np.nan, index=survey_rows.index)
        )
        if existing.notna().sum() == 0:
            existing = pd.to_numeric(parsed_response.apply(lambda d: d.get(key, np.nan)), errors="coerce")
        survey_rows[key] = existing

    trust_complete = survey_rows[trust_keys].notna().all(axis=1)
    interaction_complete = survey_rows[interaction_keys].notna().all(axis=1)

    trust_uniform = survey_rows[trust_keys].nunique(axis=1, dropna=True).eq(1)
    interaction_uniform = survey_rows[interaction_keys].nunique(axis=1, dropna=True).eq(1)

    failure_mask = trust_complete & interaction_complete & trust_uniform & interaction_uniform
    failure_rows = survey_rows[failure_mask & survey_rows["participant_id"].notna()].copy()

    if failure_rows.empty:
        return set(), pd.DataFrame()

    failure_rows["participant_id"] = failure_rows["participant_id"].astype(str)
    detail_columns = [
        "participant_id",
        "condition_id",
        "condition_number",
        *trust_keys,
        *interaction_keys,
    ]
    available_detail_columns = [col for col in detail_columns if col in failure_rows.columns]
    return set(failure_rows["participant_id"].unique()), failure_rows[available_detail_columns].reset_index(drop=True)


def normalize_scores_cousineau_morey(
    data,
    metric_columns,
    baseline_condition=1,
    drop_baseline=True,
    normalization_conditions=None,
    require_complete_participants=True,
):
    if data is None or data.empty:
        normalized = pd.DataFrame() if data is None else data.copy()
        if not normalized.empty:
            normalized[NORMALIZED_FLAG_COLUMN] = False
            normalized[MOREY_FACTOR_COLUMN] = np.nan
            normalized[K_CONDITIONS_COLUMN] = np.nan
        return normalized

    required_cols = {"participant_id", "condition_number"}
    if not required_cols.issubset(data.columns):
        normalized = data.copy()
        normalized[NORMALIZED_FLAG_COLUMN] = False
        normalized[MOREY_FACTOR_COLUMN] = np.nan
        normalized[K_CONDITIONS_COLUMN] = np.nan
        return normalized

    available_columns = [col for col in metric_columns if col in data.columns]
    if not available_columns:
        normalized = data.copy()
        normalized[NORMALIZED_FLAG_COLUMN] = False
        normalized[MOREY_FACTOR_COLUMN] = np.nan
        normalized[K_CONDITIONS_COLUMN] = np.nan
        return normalized

    working = data.copy()
    working["condition_number"] = pd.to_numeric(working["condition_number"], errors="coerce")
    working[NORMALIZED_FLAG_COLUMN] = False
    working[MOREY_FACTOR_COLUMN] = np.nan
    working[K_CONDITIONS_COLUMN] = np.nan

    if normalization_conditions is None:
        observed_conditions = working["condition_number"].dropna().astype(int).unique().tolist()
        normalization_conditions = sorted(c for c in observed_conditions if c != int(baseline_condition))
    else:
        normalization_conditions = sorted({int(c) for c in normalization_conditions})

    if not normalization_conditions:
        if drop_baseline:
            working = working[working["condition_number"] != baseline_condition].copy()
        return working.reset_index(drop=True)

    normalization_mask = working["condition_number"].isin(normalization_conditions)
    normalization_data = working[normalization_mask].copy()
    if normalization_data.empty:
        if drop_baseline:
            working = working[working["condition_number"] != baseline_condition].copy()
        return working.reset_index(drop=True)

    expected_k = len(normalization_conditions)

    if require_complete_participants:
        participant_condition_counts = (
            normalization_data.groupby("participant_id", dropna=False)["condition_number"]
            .nunique()
        )
        complete_participants = participant_condition_counts[
            participant_condition_counts >= expected_k
        ].index
        normalization_data = normalization_data[
            normalization_data["participant_id"].isin(complete_participants)
        ].copy()

    if normalization_data.empty:
        if drop_baseline:
            working = working[working["condition_number"] != baseline_condition].copy()
        return working.reset_index(drop=True)

    for col in available_columns:
        numeric_values = pd.to_numeric(normalization_data[col], errors="coerce")
        participant_means = normalization_data.groupby("participant_id", dropna=False)[col].transform(
            lambda s: pd.to_numeric(s, errors="coerce").mean()
        )
        grand_mean = numeric_values.mean()
        normalized_values = numeric_values - participant_means + grand_mean
        working.loc[normalization_data.index, col] = normalized_values

    morey_factor = np.sqrt(expected_k / (expected_k - 1)) if expected_k > 1 else np.nan
    working.loc[normalization_data.index, NORMALIZED_FLAG_COLUMN] = True
    working.loc[normalization_data.index, MOREY_FACTOR_COLUMN] = morey_factor
    working.loc[normalization_data.index, K_CONDITIONS_COLUMN] = expected_k

    if drop_baseline:
        working = working[working["condition_number"] != baseline_condition].copy()

    return working.reset_index(drop=True)


def normalize_scores_by_participant_baseline(
    data,
    metric_columns,
    treatment_conditions,
    baseline_condition=1,
    drop_baseline=True,
):
    return normalize_scores_cousineau_morey(
        data,
        metric_columns=metric_columns,
        baseline_condition=baseline_condition,
        drop_baseline=drop_baseline,
        normalization_conditions=treatment_conditions,
        require_complete_participants=True,
    )


def extract_condition_number_regex(condition_id):
    if pd.isna(condition_id):
        return np.nan
    match = re.search(r"condition_(\d+)", str(condition_id))
    return int(match.group(1)) if match else np.nan


## Experiment 1 preprocessing (from `analysis-0306.ipynb`)

In [25]:
EXP1_DATA_DIR = Path("Experiment1/data/0306/data")
EXP1_BASELINE_CONDITION_NUMBER = 1
EXP1_COUSINEAU_TREATMENT_K = 3

LIKERT_SOURCE_MIN = 0
LIKERT_SOURCE_MAX = 6
LIKERT_TARGET_MIN = 1
LIKERT_TARGET_MAX = 7


def extract_condition_number_exp1(condition_id):
    if pd.isna(condition_id):
        return np.nan
    for part in str(condition_id).split("_"):
        if part.isdigit():
            return int(part)
    return np.nan


def normalize_likert_scale_0_6_to_1_7(values):
    values = pd.to_numeric(values, errors="coerce")
    valid = values.dropna()
    if valid.empty:
        return values
    if (valid >= LIKERT_SOURCE_MIN).all() and (valid <= LIKERT_SOURCE_MAX).all():
        normalized = values + 1
        return normalized.clip(lower=LIKERT_TARGET_MIN, upper=LIKERT_TARGET_MAX)
    return values


def cousineau_normalize_within_subject_exp1(
    data,
    metric_columns,
    participant_col="participant_id",
    condition_col="condition_number",
    baseline_condition=EXP1_BASELINE_CONDITION_NUMBER,
    k_conditions=EXP1_COUSINEAU_TREATMENT_K,
):
    if data is None:
        return pd.DataFrame()

    if data.empty:
        normalized = data.copy()
        normalized[NORMALIZED_FLAG_COLUMN] = True
        normalized[MOREY_FACTOR_COLUMN] = np.sqrt(k_conditions / (k_conditions - 1)) if k_conditions > 1 else np.nan
        return normalized

    required = {participant_col, condition_col}
    if not required.issubset(data.columns):
        normalized = data.copy()
        normalized[NORMALIZED_FLAG_COLUMN] = False
        normalized[MOREY_FACTOR_COLUMN] = np.nan
        return normalized

    available_columns = [col for col in metric_columns if col in data.columns]
    if not available_columns:
        normalized = data.copy()
        normalized[NORMALIZED_FLAG_COLUMN] = False
        normalized[MOREY_FACTOR_COLUMN] = np.nan
        return normalized

    working = data.copy()
    working[condition_col] = pd.to_numeric(working[condition_col], errors="coerce")
    working = working[working[condition_col].notna()].copy()

    treatment_only = working[
        (working[condition_col] != baseline_condition)
        & (working[condition_col] != 0)
    ].copy()
    if treatment_only.empty:
        normalized = working.copy()
        normalized[NORMALIZED_FLAG_COLUMN] = False
        normalized[MOREY_FACTOR_COLUMN] = np.nan
        return normalized

    participant_condition_counts = (
        treatment_only.groupby(participant_col, dropna=False)[condition_col]
        .nunique()
    )
    complete_participants = participant_condition_counts[
        participant_condition_counts >= k_conditions
    ].index

    normalized = treatment_only[
        treatment_only[participant_col].isin(complete_participants)
    ].copy()

    if normalized.empty:
        normalized = treatment_only.copy()
        normalized[NORMALIZED_FLAG_COLUMN] = False
        normalized[MOREY_FACTOR_COLUMN] = np.nan
        return normalized

    for col in available_columns:
        normalized[col] = pd.to_numeric(normalized[col], errors="coerce")

    participant_means = (
        normalized.groupby(participant_col, dropna=False)[available_columns]
        .transform("mean")
    )
    grand_means = normalized[available_columns].mean()

    normalized[available_columns] = normalized[available_columns] - participant_means + grand_means

    morey_factor = np.sqrt(k_conditions / (k_conditions - 1)) if k_conditions > 1 else np.nan
    normalized[NORMALIZED_FLAG_COLUMN] = True
    normalized[MOREY_FACTOR_COLUMN] = morey_factor

    return normalized.reset_index(drop=True)


def load_experiment1_analysis_data(project_root=PROJECT_ROOT):
    data_dir = project_root / EXP1_DATA_DIR
    csv_files = sorted(data_dir.glob("user_*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")

    all_data = []
    for file_path in csv_files:
        df = pd.read_csv(file_path)
        df["source_file"] = file_path.name

        if "participant_id" not in df.columns or df["participant_id"].isna().all():
            participant_id = file_path.stem.split("_")[1]
            df["participant_id"] = participant_id

        if "condition_id" in df.columns:
            df["condition_id"] = df["condition_id"].fillna("unknown")
            df["condition_number"] = df["condition_id"].apply(extract_condition_number_exp1)
        else:
            df["condition_id"] = "unknown"
            df["condition_number"] = np.nan

        all_data.append(df)

    combined_data = pd.concat(all_data, ignore_index=True)

    before_participants = combined_data["participant_id"].nunique(dropna=True)
    combined_data = combined_data[combined_data["participant_id"].notna()].copy()
    combined_data = combined_data[~combined_data["participant_id"].astype(str).str.lower().eq("test")].copy()
    after_participants = combined_data["participant_id"].nunique(dropna=True)

    relevant_trial_types = [
        "prediction-task",
        "vis-literacy",
        "trust-survey",
        "personality-survey",
        "survey-text",
        "survey-multi-choice",
    ]
    relevant_trials = combined_data[combined_data["trial_type"].isin(relevant_trial_types)].copy()

    trust_survey_rows = relevant_trials[relevant_trials["trial_type"] == "trust-survey"].copy()
    trust_survey_expanded = expand_response_columns(trust_survey_rows)

    survey_likert_keys = INTERACTION_QUESTION_KEYS + TRUST_QUESTION_KEYS
    for col in survey_likert_keys:
        if col in trust_survey_expanded.columns:
            trust_survey_expanded[col] = normalize_likert_scale_0_6_to_1_7(trust_survey_expanded[col])

    analysis_metric_columns = [col for col in survey_likert_keys if col in trust_survey_expanded.columns]
    analysis_treatment_survey = cousineau_normalize_within_subject_exp1(
        trust_survey_expanded,
        metric_columns=analysis_metric_columns,
        participant_col="participant_id",
        condition_col="condition_number",
        baseline_condition=EXP1_BASELINE_CONDITION_NUMBER,
        k_conditions=EXP1_COUSINEAU_TREATMENT_K,
    )

    print("Experiment 1")
    print(f"  CSV files: {len(csv_files)}")
    print(f"  Participants: {before_participants} -> {after_participants}")
    print(f"  Trust-survey rows: {len(trust_survey_rows)}")
    print(f"  Cousineau-normalized treatment rows: {len(analysis_treatment_survey)}")
    print(f"  Participants retained for normalization: {analysis_treatment_survey['participant_id'].nunique()}")

    return analysis_treatment_survey


## Experiment 2 preprocessing (from `0325.ipynb`)

In [26]:
EXP2_DATA_DIR_CANDIDATES = [
    Path("Experiment2/data/0325/data"),
    Path("Experiment2/data/dist 11/data"),
]
EXP2_BASELINE_CONDITION_NUMBER = 1
EXP2_TREATMENT_CONDITION_NUMBERS = [2, 3, 4, 5, 6, 7, 8, 9, 10]


def load_experiment2_analysis_data(project_root=PROJECT_ROOT):
    data_dir = next((project_root / p for p in EXP2_DATA_DIR_CANDIDATES if (project_root / p).exists()), None)
    if data_dir is None:
        searched = ", ".join(str((project_root / p).resolve()) for p in EXP2_DATA_DIR_CANDIDATES)
        raise FileNotFoundError(f"No Experiment 2 data directory found. Checked: {searched}")

    csv_files = sorted(data_dir.glob("user_*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {data_dir.resolve()} using pattern 'user_*.csv'")

    all_data = []
    for file_path in csv_files:
        df = pd.read_csv(file_path)
        df["source_file"] = file_path.name

        if "participant_id" not in df.columns or df["participant_id"].isna().all():
            participant_id = file_path.stem.split("_")[1]
            df["participant_id"] = participant_id

        if "condition_id" not in df.columns:
            df["condition_id"] = np.nan

        df["condition_number"] = df["condition_id"].apply(extract_condition_number_regex)
        all_data.append(df)

    combined_data = pd.concat(all_data, ignore_index=True)

    before_participants = combined_data["participant_id"].nunique(dropna=True)
    combined_data = combined_data[combined_data["participant_id"].notna()].copy()
    combined_data = combined_data[~combined_data["participant_id"].astype(str).str.lower().eq("test")].copy()

    attention_failed_ids, _ = identify_attention_check_failures(combined_data)
    if attention_failed_ids:
        combined_data = combined_data[
            ~combined_data["participant_id"].astype(str).isin(attention_failed_ids)
        ].copy()

    after_participants = combined_data["participant_id"].nunique(dropna=True)

    trust_survey_rows = combined_data[combined_data["trial_type"] == "trust-survey"].copy()
    trust_survey_expanded = ensure_likert_columns(
        trust_survey_rows,
        TRUST_QUESTION_KEYS + INTERACTION_QUESTION_KEYS,
    )

    analysis_metric_columns = [
        col for col in (TRUST_QUESTION_KEYS + INTERACTION_QUESTION_KEYS)
        if col in trust_survey_expanded.columns
    ]

    analysis_trust_survey_expanded = normalize_scores_by_participant_baseline(
        trust_survey_expanded,
        metric_columns=analysis_metric_columns,
        treatment_conditions=EXP2_TREATMENT_CONDITION_NUMBERS,
        baseline_condition=EXP2_BASELINE_CONDITION_NUMBER,
        drop_baseline=False,
    )

    analysis_treatment_survey = analysis_trust_survey_expanded[
        analysis_trust_survey_expanded["condition_number"].isin(EXP2_TREATMENT_CONDITION_NUMBERS)
        & analysis_trust_survey_expanded[NORMALIZED_FLAG_COLUMN].fillna(False)
    ].copy()

    print("Experiment 2")
    print(f"  Data directory: {data_dir.resolve()}")
    print(f"  CSV files: {len(csv_files)}")
    print(f"  Participants: {before_participants} -> {after_participants}")
    print(f"  Attention-check exclusions: {len(attention_failed_ids)}")
    print(f"  Trust-survey rows: {len(trust_survey_rows)}")
    print(f"  Cousineau-normalized treatment rows: {len(analysis_treatment_survey)}")
    print(f"  Participants retained for normalization: {analysis_treatment_survey['participant_id'].nunique()}")

    return analysis_treatment_survey


## Experiment 3 preprocessing (from `analysis-0318.ipynb`)

In [27]:
EXP3_DATA_DIRS = [
    Path("Experiment3/data/0318/data"),
    Path("Experiment3/data/0319/data"),
    Path("Experiment3/data/0322/data"),
]
EXP3_BASELINE_CONDITION_NUMBER = 1
EXP3_TREATMENT_CONDITION_NUMBERS = [21, 22, 23, 24, 25, 26]

EXP3_INTERACTION_CONDITION_LABELS = {
    1: "Baseline",
    21: "Hover Show One",
    22: "Hover Show All",
    23: "Click Show One",
    24: "Click Show All",
    25: "Animation Show One",
    26: "Animation Show All",
}

EXP3_VISUALIZATION_TECHNIQUE_LABELS = {
    "baseline": "Baseline",
    "confidence_interval": "PI",
    "ensemble_plot": "Ensemble",
    "combined_plot": "PI + Ensemble",
}


def map_interaction_group_exp3(condition_number):
    if pd.isna(condition_number):
        return "Other"
    return EXP3_INTERACTION_CONDITION_LABELS.get(int(condition_number), "Other")


def map_visualization_group_exp3(row):
    condition_number = row.get("condition_number", np.nan)
    if pd.notna(condition_number) and int(condition_number) == EXP3_BASELINE_CONDITION_NUMBER:
        return "Baseline"

    technique_raw = str(row.get("visualization_technique", "")).strip().lower()
    if technique_raw in EXP3_VISUALIZATION_TECHNIQUE_LABELS:
        return EXP3_VISUALIZATION_TECHNIQUE_LABELS[technique_raw]

    condition_id = str(row.get("condition_id", "")).strip().lower()
    if condition_id.endswith("_ci"):
        return "PI"
    if condition_id.endswith("_ep"):
        return "Ensemble"
    if condition_id.endswith("_ciep"):
        return "PI + Ensemble"

    condition_name = str(row.get("condition_name", "")).strip().lower()
    if "baseline" in condition_name:
        return "Baseline"
    if "combined" in condition_name:
        return "PI + Ensemble"
    if "ensemble" in condition_name:
        return "Ensemble"
    if "(ci)" in condition_name or "confidence" in condition_name:
        return "PI"

    return "Other"


def load_experiment3_analysis_data(project_root=PROJECT_ROOT):
    csv_files_by_dir = {
        (project_root / data_dir): sorted((project_root / data_dir).glob("user_*.csv"))
        for data_dir in EXP3_DATA_DIRS
    }
    csv_files = [file_path for files in csv_files_by_dir.values() for file_path in files]

    if not csv_files:
        searched_dirs = ", ".join(str((project_root / path).resolve()) for path in EXP3_DATA_DIRS)
        raise FileNotFoundError(f"No CSV files found in [{searched_dirs}] using pattern 'user_*.csv'")

    all_data = []
    for data_dir, files in csv_files_by_dir.items():
        batch_label = data_dir.parent.name
        for file_path in files:
            df = pd.read_csv(file_path)
            df["source_file"] = file_path.name

            if "participant_id" not in df.columns or df["participant_id"].isna().all():
                participant_id = file_path.stem.split("_")[1]
                df["participant_id"] = participant_id

            if "condition_id" not in df.columns:
                df["condition_id"] = np.nan
            if "condition_name" not in df.columns:
                df["condition_name"] = np.nan

            df["condition_number"] = df["condition_id"].apply(extract_condition_number_regex)
            df["interaction_group"] = df["condition_number"].apply(map_interaction_group_exp3)
            df["visualization_group"] = df.apply(map_visualization_group_exp3, axis=1)
            df["data_batch"] = batch_label
            all_data.append(df)

    combined_data = pd.concat(all_data, ignore_index=True)

    before_participants = combined_data["participant_id"].nunique(dropna=True)
    combined_data = combined_data[combined_data["participant_id"].notna()].copy()
    combined_data = combined_data[~combined_data["participant_id"].astype(str).str.lower().eq("test")].copy()

    attention_failed_ids, _ = identify_attention_check_failures(combined_data)
    if attention_failed_ids:
        combined_data = combined_data[
            ~combined_data["participant_id"].astype(str).isin(attention_failed_ids)
        ].copy()

    after_participants = combined_data["participant_id"].nunique(dropna=True)

    relevant_trial_types = [
        "prediction-task",
        "vis-literacy",
        "trust-survey",
        "interaction-feedback",
        "personality-survey",
        "survey-text",
        "survey-multi-choice",
    ]

    relevant_trials = combined_data[combined_data["trial_type"].isin(relevant_trial_types)].copy()

    trust_survey_rows = relevant_trials[relevant_trials["trial_type"] == "trust-survey"].copy()
    trust_survey_expanded = ensure_likert_columns(
        trust_survey_rows,
        TRUST_QUESTION_KEYS + INTERACTION_QUESTION_KEYS,
    )

    analysis_metric_columns = [
        col for col in (TRUST_QUESTION_KEYS + INTERACTION_QUESTION_KEYS)
        if col in trust_survey_expanded.columns
    ]

    analysis_trust_survey_expanded = normalize_scores_by_participant_baseline(
        trust_survey_expanded,
        metric_columns=analysis_metric_columns,
        treatment_conditions=EXP3_TREATMENT_CONDITION_NUMBERS,
        baseline_condition=EXP3_BASELINE_CONDITION_NUMBER,
        drop_baseline=False,
    )

    analysis_treatment_survey = analysis_trust_survey_expanded[
        analysis_trust_survey_expanded["condition_number"].isin(EXP3_TREATMENT_CONDITION_NUMBERS)
        & analysis_trust_survey_expanded[NORMALIZED_FLAG_COLUMN].fillna(False)
    ].copy()

    print("Experiment 3")
    for data_dir, files in csv_files_by_dir.items():
        print(f"  Data directory: {data_dir.resolve()} ({len(files)} files)")
    print(f"  CSV files: {len(csv_files)}")
    print(f"  Participants: {before_participants} -> {after_participants}")
    print(f"  Attention-check exclusions: {len(attention_failed_ids)}")
    print(f"  Trust-survey rows: {len(trust_survey_rows)}")
    print(f"  Cousineau-normalized treatment rows: {len(analysis_treatment_survey)}")
    print(f"  Participants retained for normalization: {analysis_treatment_survey['participant_id'].nunique()}")

    return analysis_treatment_survey


## Load preprocessed analysis tables

In [28]:
exp1_analysis_data = load_experiment1_analysis_data()
exp2_analysis_data = load_experiment2_analysis_data()
exp3_analysis_data = load_experiment3_analysis_data()

experiment_data = {
    "Experiment 1 (0306)": exp1_analysis_data,
    "Experiment 2 (0325)": exp2_analysis_data,
    "Experiment 3 (0318/0319/0322)": exp3_analysis_data,
}

summary_rows = []
for exp_name, df in experiment_data.items():
    summary_rows.append(
        {
            "experiment": exp_name,
            "rows": len(df),
            "participants": df["participant_id"].nunique() if "participant_id" in df.columns else np.nan,
            "trust_complete_rows": df[TRUST_QUESTION_KEYS].notna().all(axis=1).sum() if set(TRUST_QUESTION_KEYS).issubset(df.columns) else np.nan,
            "interaction_complete_rows": df[INTERACTION_QUESTION_KEYS].notna().all(axis=1).sum() if set(INTERACTION_QUESTION_KEYS).issubset(df.columns) else np.nan,
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


Experiment 1
  CSV files: 337
  Participants: 337 -> 337
  Trust-survey rows: 1340
  Cousineau-normalized treatment rows: 1005
  Participants retained for normalization: 335
Experiment 2
  Data directory: /Users/songwen/Documents/Code Lab/Interact4Trust/Experiment2/data/0325/data
  CSV files: 91
  Participants: 91 -> 87
  Attention-check exclusions: 4
  Trust-survey rows: 870
  Cousineau-normalized treatment rows: 783
  Participants retained for normalization: 87
Experiment 3
  Data directory: /Users/songwen/Documents/Code Lab/Interact4Trust/Experiment3/data/0318/data (27 files)
  Data directory: /Users/songwen/Documents/Code Lab/Interact4Trust/Experiment3/data/0319/data (57 files)
  Data directory: /Users/songwen/Documents/Code Lab/Interact4Trust/Experiment3/data/0322/data (58 files)
  CSV files: 142
  Participants: 142 -> 140
  Attention-check exclusions: 2
  Trust-survey rows: 980
  Cousineau-normalized treatment rows: 840
  Participants retained for normalization: 140


,experiment,rows,participants,trust_complete_rows,interaction_complete_rows
0,Experiment 1 (0306),1005,335,1005,1005
1,Experiment 2 (0325),783,87,783,783
2,Experiment 3 (0318/0319/0322),840,140,840,840


## Fixed factor analyses (1, 2, 4 factors)

In [29]:
def prepare_factor_rows(data, question_keys, section_label):
    if data is None or data.empty:
        print(f"[skip] {section_label}: no rows available.")
        return pd.DataFrame()

    available_keys = [k for k in question_keys if k in data.columns]
    if len(available_keys) < 2:
        print(f"[skip] {section_label}: fewer than 2 usable columns.")
        return pd.DataFrame()

    analysis_rows = data[available_keys].apply(pd.to_numeric, errors="coerce").dropna()

    if analysis_rows.shape[0] < 10 or analysis_rows.shape[1] < 2:
        print(
            f"[skip] {section_label}: insufficient complete rows/columns "
            f"(n={analysis_rows.shape[0]}, p={analysis_rows.shape[1]})."
        )
        return pd.DataFrame()

    constant_columns = [col for col in analysis_rows.columns if analysis_rows[col].nunique(dropna=True) <= 1]
    if constant_columns:
        analysis_rows = analysis_rows.drop(columns=constant_columns)
        print(f"[note] {section_label}: dropped constant columns: {', '.join(constant_columns)}")

    if analysis_rows.shape[1] < 2:
        print(f"[skip] {section_label}: too few non-constant columns.")
        return pd.DataFrame()

    return analysis_rows


def run_fixed_factor_analysis(
    data,
    question_keys,
    section_label,
    n_factors,
    threshold=LOADING_THRESHOLD,
    prompt_map=None,
):
    analysis_rows = prepare_factor_rows(data, question_keys, section_label)
    if analysis_rows.empty:
        return {}

    max_components = min(analysis_rows.shape[1], analysis_rows.shape[0] - 1)
    if n_factors > max_components:
        print(
            f"[skip] {section_label} ({n_factors} factors): requested components exceed "
            f"max allowable ({max_components})."
        )
        return {}

    fa = FactorAnalysis(n_components=n_factors, random_state=42)
    factor_scores = fa.fit_transform(analysis_rows)

    factor_cols = [f"Factor {i + 1}" for i in range(n_factors)]
    loading_matrix = pd.DataFrame(
        fa.components_.T,
        index=analysis_rows.columns,
        columns=factor_cols,
    )

    if isinstance(prompt_map, dict) and prompt_map:
        loading_matrix.index = [prompt_map.get(k, k) for k in loading_matrix.index]

    print()
    print(f"=== {section_label} | {n_factors} factor(s) ===")
    print(f"Rows used: {len(analysis_rows)}")
    print(f"Variables tested: {analysis_rows.shape[1]}")

    filtered_table = loading_matrix.where(loading_matrix.abs() >= threshold)
    filtered_table["max_abs_loading"] = loading_matrix.abs().max(axis=1)
    filtered_table = filtered_table[filtered_table[factor_cols].notna().any(axis=1)]
    filtered_table = filtered_table.sort_values("max_abs_loading", ascending=False)

    return {
        "n_factors": n_factors,
        "loadings": loading_matrix,
        "filtered_loadings": filtered_table,
        "scores": pd.DataFrame(factor_scores, columns=factor_cols),
        "analysis_rows": analysis_rows,
    }


def run_factor_suite_for_experiment(experiment_label, data):
    print("\n" + "=" * 100)
    print(f"{experiment_label}")

    results = {}

    for n_factors in FACTOR_COUNTS:
        results[f"trust_{n_factors}f"] = run_fixed_factor_analysis(
            data,
            TRUST_QUESTION_KEYS,
            section_label=f"{experiment_label} - Trust / Usability block",
            n_factors=n_factors,
            threshold=LOADING_THRESHOLD,
            prompt_map=TRUST_PROMPT_MAP,
        )

    for n_factors in FACTOR_COUNTS:
        results[f"interaction_{n_factors}f"] = run_fixed_factor_analysis(
            data,
            INTERACTION_QUESTION_KEYS,
            section_label=f"{experiment_label} - Interaction block",
            n_factors=n_factors,
            threshold=LOADING_THRESHOLD,
            prompt_map=INTERACTION_PROMPT_MAP,
        )

    return results


def _experiment_short_label(experiment_name):
    match = re.search(r"Experiment\s*(\d+)", str(experiment_name))
    if match:
        return f"Exp{match.group(1)}"
    return str(experiment_name)


def _factor_model_label(n_factors):
    return f"{n_factors}-factor"


def build_combined_loading_table(all_factor_results, threshold=LOADING_THRESHOLD):
    row_index = [
        *[f"Trust | {TRUST_PROMPT_MAP.get(k, k)}" for k in TRUST_QUESTION_KEYS],
        *[f"Interaction | {INTERACTION_PROMPT_MAP.get(k, k)}" for k in INTERACTION_QUESTION_KEYS],
    ]

    factor_labels_by_model = {
        1: ["factor1"],
        2: ["factor1", "factor2"],
        4: ["factor1", "factor2", "factor3", "factor4"],
    }

    experiment_labels = [_experiment_short_label(name) for name in all_factor_results.keys()]
    column_tuples = [
        (exp_label, _factor_model_label(n_factors), factor_label)
        for exp_label in experiment_labels
        for n_factors in FACTOR_COUNTS
        for factor_label in factor_labels_by_model.get(n_factors, [])
    ]

    columns = pd.MultiIndex.from_tuples(
        column_tuples,
        names=["Experiment", "Factor Model", "Factor"],
    )

    combined_table = pd.DataFrame(np.nan, index=row_index, columns=columns, dtype=float)

    for experiment_name, experiment_results in all_factor_results.items():
        exp_label = _experiment_short_label(experiment_name)

        for block_name, block_label in [("trust", "Trust"), ("interaction", "Interaction")]:
            for n_factors in FACTOR_COUNTS:
                result_key = f"{block_name}_{n_factors}f"
                result = experiment_results.get(result_key, {})
                loading_matrix = result.get("loadings") if isinstance(result, dict) else None

                if not isinstance(loading_matrix, pd.DataFrame) or loading_matrix.empty:
                    continue

                for factor_idx, factor_col in enumerate(loading_matrix.columns, start=1):
                    target_column = (
                        exp_label,
                        _factor_model_label(n_factors),
                        f"factor{factor_idx}",
                    )
                    if target_column not in combined_table.columns:
                        continue

                    for question_text, loading_value in loading_matrix[factor_col].items():
                        target_row = f"{block_label} | {question_text}"
                        if target_row in combined_table.index:
                            combined_table.loc[target_row, target_column] = loading_value

    combined_table = combined_table.where(combined_table.abs() >= threshold)
    return combined_table


def export_combined_table_to_latex(combined_table, output_path="factor.tex", decimals=3):
    rounded = combined_table.round(decimals)
    latex = rounded.to_latex(
        na_rep="",
        float_format=lambda x: f"{x:.{decimals}f}",
        multicolumn=True,
        multirow=True,
        longtable=True,
        escape=True,
    )
    output_file = Path(output_path)
    output_file.write_text(latex)
    return output_file


In [30]:
all_factor_results = {}
for exp_name, df in experiment_data.items():
    all_factor_results[exp_name] = run_factor_suite_for_experiment(exp_name, df)

combined_factor_loading_table = build_combined_loading_table(
    all_factor_results,
    threshold=LOADING_THRESHOLD,
)
print("\nCombined factor-loading table (trust + interaction questions)")
display(combined_factor_loading_table.round(3))

latex_output_path = export_combined_table_to_latex(
    combined_factor_loading_table,
    output_path="factor.tex",
    decimals=3,
)
print(f"LaTeX table written to: {latex_output_path.resolve()}")



Experiment 1 (0306)

=== Experiment 1 (0306) - Trust / Usability block | 1 factor(s) ===
Rows used: 1005
Variables tested: 4

=== Experiment 1 (0306) - Trust / Usability block | 2 factor(s) ===
Rows used: 1005
Variables tested: 4

=== Experiment 1 (0306) - Trust / Usability block | 4 factor(s) ===
Rows used: 1005
Variables tested: 4

=== Experiment 1 (0306) - Interaction block | 1 factor(s) ===
Rows used: 1005
Variables tested: 9

=== Experiment 1 (0306) - Interaction block | 2 factor(s) ===
Rows used: 1005
Variables tested: 9

=== Experiment 1 (0306) - Interaction block | 4 factor(s) ===
Rows used: 1005
Variables tested: 9

Experiment 2 (0325)

=== Experiment 2 (0325) - Trust / Usability block | 1 factor(s) ===
Rows used: 783
Variables tested: 4

=== Experiment 2 (0325) - Trust / Usability block | 2 factor(s) ===
Rows used: 783
Variables tested: 4

=== Experiment 2 (0325) - Trust / Usability block | 4 factor(s) ===
Rows used: 783
Variables tested: 4

=== Experiment 2 (0325) - Interac

Experiment                                             Exp1                   \
Factor Model                                       1-factor 2-factor           
Factor                                              factor1  factor1 factor2   
Trust | I was skeptical of the visualization ou...   -0.428   -0.458     NaN   
Trust | I trusted the data shown by the visuali...    0.435    0.534  -0.377   
Trust | The visualization was difficult to use.      -0.918   -0.949  -0.413   
Trust | The visualization made the task easier ...    0.846    0.779     NaN   
Interaction | I was in control of my navigation...   -0.481   -0.494     NaN   
Interaction | I had control over the content I ...   -0.651   -0.828   0.340   
Interaction | I was in control of the pace of m...   -0.408   -0.401     NaN   
Interaction | The interface supported the way I...   -0.556   -0.549     NaN   
Interaction | The interface responded quickly a...   -0.520   -0.467     NaN   
Interaction | The interface supported communica...      NaN      NaN     NaN   
Interaction | Interacting with the interface fe...   -0.360   -0.312     NaN   
Interaction | I felt that I could interact mean...   -0.445   -0.401     NaN   
Interaction | The interface felt sensitive to m...   -0.481   -0.442     NaN   

Experiment                                                                   \
Factor Model                                       4-factor                   
Factor                                              factor1 factor2 factor3   
Trust | I was skeptical of the visualization ou...   -0.450     NaN     NaN   
Trust | I trusted the data shown by the visuali...    0.443     NaN     NaN   
Trust | The visualization was difficult to use.      -0.878     NaN     NaN   
Trust | The visualization made the task easier ...    0.763     NaN     NaN   
Interaction | I was in control of my navigation...   -0.506     NaN     NaN   
Interaction | I had control over the content I ...   -0.704   0.386     NaN   
Interaction | I was in control of the pace of m...   -0.412     NaN     NaN   
Interaction | The interface supported the way I...   -0.567     NaN   0.304   
Interaction | The interface responded quickly a...   -0.520     NaN     NaN   
Interaction | The interface supported communica...      NaN     NaN     NaN   
Interaction | Interacting with the interface fe...   -0.367     NaN     NaN   
Interaction | I felt that I could interact mean...   -0.447     NaN     NaN   
Interaction | The interface felt sensitive to m...   -0.468     NaN     NaN   

Experiment                                                     Exp2           \
Factor Model                                               1-factor 2-factor   
Factor                                             factor4  factor1  factor1   
Trust | I was skeptical of the visualization ou...     NaN   -0.564   -0.597   
Trust | I trusted the data shown by the visuali...     NaN    0.471    0.500   
Trust | The visualization was difficult to use.        NaN   -0.686   -0.696   
Trust | The visualization made the task easier ...     NaN    0.680    0.684   
Interaction | I was in control of my navigation...     NaN   -0.353   -0.472   
Interaction | I had control over the content I ...     NaN   -0.330   -0.372   
Interaction | I was in control of the pace of m...     NaN      NaN   -0.303   
Interaction | The interface supported the way I...     NaN   -0.346   -0.314   
Interaction | The interface responded quickly a...     NaN   -0.440   -0.381   
Interaction | The interface supported communica...     NaN      NaN      NaN   
Interaction | Interacting with the interface fe...     NaN   -0.301      NaN   
Interaction | I felt that I could interact mean...     NaN   -0.304      NaN   
Interaction | The interface felt sensitive to m...     NaN      NaN      NaN   

Experiment                                                                   \
Factor Model                                               4-factor           
Factor         

LaTeX table written to: /Users/songwen/Documents/Code Lab/Interact4Trust/factor.tex
